In [17]:
import pandas as pd
import geopandas as gpd

In [18]:
from shapely.geometry import Polygon, MultiPolygon, GeometryCollection
from shapely.ops import unary_union

def extract_polygon_geom(geom):
    
    """Extract Polygon/MultiPolygon geometries from a GeometryCollection"""

    if geom is None:
        return None
    
    elif geom.geom_type in ['Polygon', 'MultiPolygon']:
        return geom
    
    elif geom.geom_type == 'GeometryCollection':
        polygons = [g for g in geom.geoms if g.geom_type in ['Polygon', 'MultiPolygon']]

        if not polygons:
            return None
        
        return unary_union(polygons)  # Merge into a MultiPolygon or Polygon
    
    else:
        return None  # Ignore Point, LineString, etc.

In [19]:
def population_weighted_variables_mapping(
    shp_source_path, shp_target_path,
    data_csv, population_csv,
    source_id_col,
    target_id_col,
    variable_fields,
    population_field
):
    """
    Project variables from a source LSOA geometry (2001 or 2011) to 2021 LSOA geometry
    using population-weighted interpolation based entirely on spatial overlay.
    
    Parameters:
    - shp_source_path: path to source LSOA shapefile
    - shp_target_path: path to target LSOA shapefile
    - data_csv: CSV with variables to project
    - population_csv: CSV with population per source LSOA
    - source_id_col: column name for source LSOA ID
    - target_id_col: column name for target LSOA ID
    - variable_fields: list of variable fields to interpolate
    - population_field: name of population field in population_csv

    Returns:
    - DataFrame with [target_id_col] and interpolated variables
    """
    # Load shapefiles and data
    gdf_src = gpd.read_file(shp_source_path)
    gdf_tgt = gpd.read_file(shp_target_path)
    data_df = pd.read_csv(data_csv)
    pop_df = pd.read_csv(population_csv)

    # Clean column names
    data_df.columns = data_df.columns.str.replace('\xa0', ' ', regex=True).str.strip()
    pop_df.columns = pop_df.columns.str.replace('\xa0', ' ', regex=True).str.strip()

    # Keep necessary columns
    data_df = data_df[[source_id_col] + variable_fields]
    pop_df = pop_df[[source_id_col, population_field]]

    # Merge attributes into source GeoDataFrame
    gdf_src = gdf_src.merge(data_df, on=source_id_col, how='left')
    gdf_src = gdf_src.merge(pop_df, on=source_id_col, how='left')

    # Check required columns
    required_cols = [source_id_col, population_field] + variable_fields
    
    for col in required_cols:
        if col not in gdf_src.columns:
            raise KeyError(f"Missing column '{col}' in source GeoDataFrame.")

    # Ensure matching CRS
    gdf_tgt = gdf_tgt.to_crs(gdf_src.crs)

    # Compute source area
    gdf_src_clone = gdf_src.copy()
    gdf_src['area_total'] = gdf_src_clone.geometry.area

    # Spatial overlay — keep all geometries to avoid loss
    intersections = gpd.overlay(gdf_src, gdf_tgt, how='intersection', keep_geom_type=False)

    intersections['geometry'] = intersections.geometry.apply(extract_polygon_geom)
    intersections = intersections[intersections.geometry.notnull()]

    intersections['area_ij'] = intersections.geometry.area

    intersections['pop_ij'] = (intersections['area_ij'] / intersections['area_total']) * intersections[population_field]

    for field in variable_fields:
        intersections[field] = intersections[field].fillna(0)
        intersections[f'{field}_weighted'] = intersections['pop_ij'] / intersections[population_field] * intersections[field]

    agg_fields = {f'{field}_weighted': 'sum' for field in variable_fields}
    agg_fields['pop_ij'] = 'sum'

    result = intersections.groupby(target_id_col).agg(agg_fields).reset_index()

    result.drop(columns=['pop_ij'], inplace=True)

    return result

**Convert data of property sales**

In [ ]:
df_num_of_sales = population_weighted_variables_mapping(
    shp_source_path = "../Output/London_LSOA_2011.shp",
    shp_target_path = "../Output/London_LSOA_2021.shp",
    data_csv = "../Dataset/Other Datasets/Number of sales.csv",
    population_csv = "../Output/Population_2011.csv",
    source_id_col = "LSOA11CD",
    target_id_col = "LSOA21CD",
    variable_fields = ["Number of sales 2011", "Number of sales 2021"],
    population_field = "All residents"
)

df_num_of_sales = df_num_of_sales.round(0)

In [22]:
df_num_of_sales.to_csv("../Dataset/Other Datasets/Number of sales mapped.csv", index=False)

**Convert IMD to census 2021 LSOA scale**

In [23]:
df_imd_2019_ = population_weighted_variables_mapping(
    shp_source_path = "../Output/London_LSOA_2011.shp",
    shp_target_path = "../Output/London_LSOA_2021.shp",
    data_csv = "../Dataset/IMD/IMD_london_2019.csv",
    population_csv = "../Output/Population_2011.csv",
    source_id_col = "LSOA11CD",
    target_id_col = "LSOA21CD",
    variable_fields = ["Index of Multiple Deprivation (IMD) Score"],
    population_field = "All residents"
)

In [24]:
df_imd_2019_.to_csv("../Dataset/IMD/IMD2019 mapped.csv", index=False)

In [25]:
df_imd_2010_ = population_weighted_variables_mapping(
    shp_source_path = "../Output/London_LSOA_2001.shp",
    shp_target_path = "../Output/London_LSOA_2021.shp",
    data_csv = "../Dataset/IMD/IMD_london_2010.csv",
    population_csv = "../Output/Population_2001.csv",
    source_id_col = "LSOA01CD",
    target_id_col = "LSOA21CD",
    variable_fields = ["IMD SCORE"],
    population_field = "All people"
)

In [26]:
df_imd_2010_.to_csv("../Dataset/IMD/IMD2010 mapped.csv", index=False)

**Convert all the data in census 2011 to census 2021**

In [27]:
import os

# Define paths
data_dir = "../Dataset/Census London 2011/" 
population_csv = "../Output/Population_2011.csv" 
shp_source_path = "../Output/London_LSOA_2011.shp"
shp_target_path = "../Output/London_LSOA_2021.shp" 
source_id_col = "LSOA11CD"
target_id_col = "LSOA21CD"
population_field = "All residents"

# Dictionary to store results for each file
results = {}

# Loop over all CSV files ending with "2011.csv"
for filename in os.listdir(data_dir):
    if filename.endswith("2011.csv"):
        filepath = os.path.join(data_dir, filename)

        # Read the CSV to inspect the columns
        df = pd.read_csv(filepath)

        # Get all columns from the third one onward (i.e., exclude LSOA code and possibly Borough name)
        variable_fields = list(df.columns[2:])

        # Apply the population-weighted variable mapping function
        mapped_df = population_weighted_variables_mapping(
            shp_source_path=shp_source_path,
            shp_target_path=shp_target_path,
            data_csv=filepath,
            population_csv=population_csv,
            source_id_col=source_id_col,
            target_id_col=target_id_col,
            variable_fields=variable_fields,
            population_field=population_field
        )

        # Round the results to 0 decimal places
        mapped_df = mapped_df.round(0)

        # Store the processed DataFrame in the results dictionary
        results[filename] = mapped_df

c:\Users\wbwha\AppData\Local\Programs\Python\Python313\Lib\site-packages\geopandas\geodataframe.py:1819: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
c:\Users\wbwha\AppData\Local\Programs\Python\Python313\Lib\site-packages\geopandas\geodataframe.py:1819: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
c:\Users\wbwha\AppData\Local\Programs\Python\Python313\Lib\site-packages\geopandas\geodataframe.py:1819: PerformanceWarning: DataFrame is highly fragmented.  This is us

In [28]:
# Output folder to save the processed files
output_dir = "../Dataset/Census London 2011/"

# Save each result to a new CSV
for filename, df in results.items():
    output_filename = filename.replace(".csv", " mapped.csv")
    df.to_csv(os.path.join(output_dir, output_filename), index=False)